In [0]:
dbutils.fs.mkdirs("dbfs:/FileStore/streaming_csv_input")

Out[3]: True

In [0]:
%fs ls /FileStore/streaming_csv_input

In [0]:
streaming_input_path = "dbfs:/FileStore/streaming_csv_input/"

from pyspark.sql.types import StructType, StringType, IntegerType, DoubleType

schema = StructType() \
    .add("Name", StringType()) \
    .add("Age", IntegerType()) \
    .add("City", StringType()) \
    .add("Salary", DoubleType())
	
df_stream = (
    spark.readStream
    .option("header", True)
    .schema(schema)
    .csv(streaming_input_path)
)

In [0]:
from pyspark.sql.types import StructType, StringType, IntegerType

# Define schema of your CSV
schema = StructType() \
    .add("Name", StringType()) \
    .add("Age", IntegerType()) \
    .add("City", StringType()) \
    .add("Salary", IntegerType())

# Read CSV in streaming mode
streaming_df = spark.readStream \
    .option("header", "true") \
    .schema(schema) \
    .csv("dbfs:/FileStore/streaming_csv_input")

In [0]:
query = streaming_df.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("people_stream") \
    .start()


csv_content = """Name,Age,City,Salary
Tom,35,Chicago,80000
Sarah,29,Austin,72000
"""

with open("/tmp/stream_test.csv", "w") as f:
    f.write(csv_content)

dbutils.fs.cp("file:/tmp/stream_test.csv", "dbfs:/FileStore/streaming_csv_input/stream_test1.csv")

Out[6]: True

In [0]:
%sql
-- SQL cell or use spark.sql in Python
SELECT * FROM people_stream

Name,Age,City,Salary
Tom,35,Chicago,80000
Sarah,29,Austin,72000


In [0]:
query.stop()